# FGVC-Aircraft: are there copied photos?

The research uses only data without duplicates. A copy of a photo on both
sides of a split tests the model on an image it has seen. BreastMNIST held
150 groups of copies (`../../medmnist_breastmnist/demos/copies.ipynb`), so
this project's task on FGVC-Aircraft, 1,200 photos of the Boeing 737 and
the Airbus A320 family, gets the same check.

`research.ipynb` compared 64-bit difference hashes and found no copies. A
hash of a 9 x 8 thumbnail is a weak test: it cannot match a cropped or
rescaled copy, and it confuses different aircraft against a similar sky.
This notebook uses the method that found BreastMNIST's copies
(`../../medmnist_breastmnist/copies.py`), unchanged:

- for each photo, its 30 nearest photos by ResNet-50 embedding are the
  candidates;
- SIFT keypoints are matched between the two (Lowe's ratio, 0.75);
- a pair is a copy when the matches fit one affine transform (RANSAC,
  2 pixels) with at least 15 inliers, the overlap covers a fifth of the
  photo, and the warped photo correlates with the other above 0.9.

The photos are for non-commercial research only, so this notebook shows
none of them.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT))  # repo root, so featsel_datasets imports

from featsel_datasets.fgvc_aircraft.aircraft import FGVCAircraft
from featsel_datasets.fgvc_aircraft.featsel_view import load_xy
from featsel_datasets.medmnist_breastmnist.copies import (
    MIN_CORRELATION, MIN_INLIERS, N_CANDIDATES, copy_groups, copy_pairs)

data = FGVCAircraft()
X, y = load_xy(data)
print(f'{len(y)} photos; candidates {N_CANDIDATES}, inliers >= {MIN_INLIERS}, '
      f'correlation > {MIN_CORRELATION}')

1200 photos; candidates 30, inliers >= 15, correlation > 0.9


## The photos

The archive is read once, without unpacking it (about a minute). Each
photo loses its copyright banner, becomes grayscale, and is resized so its
longer side is 448 pixels, twice the 224 of BreastMNIST's images. Most
photos are about 1,000 pixels wide; the smaller size keeps the search to
minutes and leaves enough detail for SIFT.

In [2]:
LONG_SIDE = 448

start = time.perf_counter()
photos = data.images(list(y.index))
images = []
for photo in photos.values():
    scale = LONG_SIDE / max(photo.size)
    size = (round(photo.width * scale), round(photo.height * scale))
    images.append(np.asarray(photo.convert('L').resize(size)))
del photos
sizes = pd.DataFrame([im.shape for im in images], columns=['height', 'width'])
print(f'read in {time.perf_counter() - start:.0f} s')
sizes.describe().loc[['min', '50%', 'max']].astype(int)

read in 19 s


,height,width
min,229,448
50%,296,448
max,333,448


## The search

In [3]:
start = time.perf_counter()
pairs = copy_pairs(images, X.to_numpy())
print(f'{len(pairs)} pairs of copies, found in {time.perf_counter() - start:.0f} s')

3 pairs of copies, found in 79 s


In [4]:
if len(pairs):
    ids = y.index.to_numpy()
    shown = pairs.assign(
        first=ids[pairs['first']], second=ids[pairs['second']],
        first_label=y.to_numpy()[pairs['first']], second_label=y.to_numpy()[pairs['second']],
        first_variant=data.samples.loc[ids[pairs['first']], 'variant'].to_numpy(),
        second_variant=data.samples.loc[ids[pairs['second']], 'variant'].to_numpy())
    groups = copy_groups(pairs, len(y))
    print(f'{groups.nunique()} groups holding {len(groups)} photos; '
          f'{(shown.first_label != shown.second_label).sum()} pairs with different labels')
    display(shown.round(2))

1 groups holding 3 photos; 0 pairs with different labels


,first,second,scale,shift,first_label,second_label,first_variant,second_variant
0,0223783,0221294,0.98,19.39,Boeing 737,Boeing 737,737-900,737-900
1,0223783,0227126,0.99,10.86,Boeing 737,Boeing 737,737-900,737-900
2,0221294,0227126,1.01,8.67,Boeing 737,Boeing 737,737-900,737-900


## Does the search find copies in these photos?

A search that finds nothing could also be one that cannot find anything.
So 20 photos, drawn at random, each get a made copy: cropped by 10% on
every side and rescaled to 90%, as BreastMNIST's copies were. The search's
own test, `copy_transform`, then compares each photo with its copy.

In [5]:
import cv2

from featsel_datasets.medmnist_breastmnist.copies import copy_transform

sift = cv2.SIFT.create()
matcher = cv2.BFMatcher(cv2.NORM_L2)
rng = np.random.default_rng(42)
found = []
for i in rng.choice(len(images), 20, replace=False):
    im = images[i]
    h, w = im.shape
    crop = im[h // 10: h - h // 10, w // 10: w - w // 10]
    made = cv2.resize(crop, (round(crop.shape[1] * 0.9), round(crop.shape[0] * 0.9)))
    affine = copy_transform(cv2, matcher, im, made, sift.detectAndCompute(im, None),
                            sift.detectAndCompute(made, None))
    found.append(affine is not None)
print(f'made copies found: {sum(found)} of {len(found)}')

made copies found: 20 of 20


## How close did the other candidates come?

Many candidate pairs have 15 or more matches that fit one affine
transform: the same airline, at the same airport, in a similar pose. They
fail on the last test, the correlation of the aligned photos. The table
shows, for every pair with at least 15 inliers, that correlation, and the
pairs closest to the 0.9 limit.

In [6]:
features = [sift.detectAndCompute(im, None) for im in images]
unit = X.to_numpy() / np.linalg.norm(X.to_numpy(), axis=1, keepdims=True)
similarity = unit @ unit.T
np.fill_diagonal(similarity, -np.inf)
candidates = {tuple(sorted((i, int(j)))) for i in range(len(images))
              for j in np.argsort(-similarity[i])[:N_CANDIDATES]}

rows = []
for i, j in sorted(candidates):
    (ka, da), (kb, db) = features[i], features[j]
    good = [m for m, n in (p for p in matcher.knnMatch(da, db, k=2) if len(p) == 2)
            if m.distance < 0.75 * n.distance]
    if len(good) < MIN_INLIERS:
        continue
    src = np.float32([ka[m.queryIdx].pt for m in good])
    dst = np.float32([kb[m.trainIdx].pt for m in good])
    affine, mask = cv2.estimateAffine2D(src, dst, ransacReprojThreshold=2.0)
    if affine is None or mask.sum() < MIN_INLIERS:
        continue
    size = images[j].shape[::-1]
    warped = cv2.warpAffine(images[i], affine, size)
    overlap = cv2.warpAffine(np.full_like(images[i], 255), affine, size) == 255
    a, b = warped[overlap].astype(float), images[j][overlap].astype(float)
    if a.std() > 0 and b.std() > 0:
        rows.append({'first': y.index[i], 'second': y.index[j],
                     'variant': data.samples.loc[y.index[i], 'variant'],
                     'inliers': int(mask.sum()), 'overlap': overlap.mean(),
                     'correlation': np.corrcoef(a, b)[0, 1]})
near = pd.DataFrame(rows).sort_values('correlation', ascending=False)
print(f'{len(candidates)} candidate pairs; {len(near)} with >= {MIN_INLIERS} inliers')
print(near['correlation'].describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_string())
near.head(10).round(3)

/home/dror/projects/featsel/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/dror/projects/featsel/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/dror/projects/featsel/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


30109 candidate pairs; 607 with >= 15 inliers
count    607.000
mean       0.404
std        0.270
min       -0.758
50%        0.455
90%        0.723
99%        0.860
max        0.940


,first,second,variant,inliers,overlap,correlation
233,0223783,0227126,737-900,79,0.942,0.940
460,0221294,0227126,737-900,73,0.991,0.906
232,0223783,0221294,737-900,62,0.854,0.903
91,0928043,1648294,737-600,78,0.732,0.873
451,0276022,0311908,737-900,40,0.954,0.871
165,1818747,1795553,737-900,136,0.968,0.861
472,0671304,0275806,737-900,70,0.897,0.861
407,0956273,1401878,737-600,65,0.985,0.854
170,1016538,0467798,737-900,75,0.843,0.847
468,0671304,0305545,737-900,68,0.929,0.825


## Is the embedding distance alone enough?

A simpler test would compare embeddings only: the distance between two
photos' rows, divided by the RMS length of the two rows. Near-copies should
have a much smaller distance than other pairs. The table places the group
found above among all 719,400 pairs.

In [7]:
E = X.to_numpy().astype(np.float64)
squared = (E ** 2).sum(axis=1)
distance = np.sqrt(np.maximum(squared[:, None] + squared[None, :] - 2 * E @ E.T, 0))
relative = distance / np.sqrt((squared[:, None] + squared[None, :]) / 2)
upper = np.triu_indices_from(relative, 1)
others = relative[upper]
group = [y.index.get_loc(i) for i in ('0223783', '0221294', '0227126')]
rows = {}
for a, b in [(0, 1), (0, 2), (1, 2)]:
    r = relative[group[a], group[b]]
    rows[f'{y.index[group[a]]} - {y.index[group[b]]}'] = {
        'relative distance': r, 'pairs as close or closer': int((others <= r).sum()) - 1}
print(f'all pairs: smallest {others.min():.3f}, 1st percentile {np.percentile(others, 1):.3f}, '
      f'median {np.median(others):.3f}')
print(f'pairs at most 1e-6 apart (identical files): {(others <= 1e-6).sum()}')
pd.DataFrame(rows).T.round(3)

all pairs: smallest 0.139, 1st percentile 0.288, median 0.473
pairs at most 1e-6 apart (identical files): 0


,relative distance,pairs as close or closer
0223783 - 0221294,0.171,5.0
0223783 - 0227126,0.207,79.0
0221294 - 0227126,0.224,259.0


No pair is closer than 1e-6, the rounding of 32-bit floats, so no two
photos are the same file. Such a test finds only identical files: a copy
shifted or rescaled by even a few pixels moves the embedding far more.

The group found lies at 0.17 to 0.22, but 5 to 259 other pairs are as
close or closer. Looked at by eye (not shown, for the licence), the closest
pairs are the same aircraft type in the same pose against a plain sky, some
of them perhaps the same airframe on another day; none is a copy. On
BreastMNIST the measure behaves the same way: below 0.26 it finds half of
the known copies and no other pair, but some copies lie at 0.77, where
92,575 other pairs lie too. So the embedding distance alone cannot decide.
It serves well as the first step: the 30 nearest photos by embedding are
the candidates that the registration test then checks.

## Conclusion

FGVC-Aircraft's 1,200 photos hold no copied files, but one group of three
near-duplicates.

- **The search works on these photos:** it found all 20 made copies.
- **One group, three photos** (`0221294`, `0223783`, `0227126`): one
  Continental 737-900, photographed seconds apart as it taxis. The photos
  are shifted by 9 to 19 pixels and correlate at 0.90 to 0.94 once aligned.
  All three have the same label.
- **No other pair comes close:** the next pairs correlate at 0.87 or
  less. They are the same airline at the same airport in a similar pose.
- **Not checked:** the same airframe photographed on different days. The
  dataset gives no registration numbers, and such photos differ in
  background, light and pose, so this test does not find them.

To keep only data without duplicates, two of the three photos would be
dropped, leaving 1,198.